***type something cool later***

*notes:*

target is 5,174 (active) : 1,869 (churned).. class_weight='balanced'

In [1]:
import pandas as pd
import numpy as np
import math as mt
import warnings

data_contract = pd.read_csv('contract.csv')
data_internet = pd.read_csv('internet.csv')
data_personal = pd.read_csv('personal.csv')
data_phone = pd.read_csv('phone.csv')

display(data_contract.head())
display(data_internet.head())
display(data_personal.head())
display(data_phone.head())

,customerID,BeginDate,EndDate,Type,PaperlessBilling,PaymentMethod,MonthlyCharges,TotalCharges
0,7590-VHVEG,2020-01-01,No,Month-to-month,Yes,Electronic check,29.85,29.85
1,5575-GNVDE,2017-04-01,No,One year,No,Mailed check,56.95,1889.5
2,3668-QPYBK,2019-10-01,2019-12-01 00:00:00,Month-to-month,Yes,Mailed check,53.85,108.15
3,7795-CFOCW,2016-05-01,No,One year,No,Bank transfer (automatic),42.30,1840.75
4,9237-HQITU,2019-09-01,2019-11-01 00:00:00,Month-to-month,Yes,Electronic check,70.70,151.65


,customerID,InternetService,OnlineSecurity,OnlineBackup,DeviceProtection,TechSupport,StreamingTV,StreamingMovies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No


,customerID,gender,SeniorCitizen,Partner,Dependents
0,7590-VHVEG,Female,0,Yes,No
1,5575-GNVDE,Male,0,No,No
2,3668-QPYBK,Male,0,No,No
3,7795-CFOCW,Male,0,No,No
4,9237-HQITU,Female,0,No,No


,customerID,MultipleLines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes


In [2]:
    #!! converting date columns to datetime format
warnings.filterwarnings('ignore')
data_contract['BeginDate'] = pd.to_datetime(data_contract['BeginDate'])
data_contract['EndDate'] = pd.to_datetime(data_contract['EndDate'], errors='coerce')
warnings.filterwarnings('default')

    #!! filling NaT (EndDate = 'No') with 'current day' (data set extraction date 2020-02-01)
data_contract['EndDate'] = data_contract['EndDate'].fillna(pd.Timestamp('2020-02-01'))

    #!! calculating tenure in months
data_contract['tenure_month'] = (data_contract['EndDate'] - data_contract['BeginDate']).dt.days // 30

    #!! dropping BeginDate (no relationship with customer loyalty) & TotalCharges (can infer from tenure * monthly charges)
data_contract = data_contract.drop(['BeginDate', 'TotalCharges'], axis=1)

    #!! converting EndDate to boolean: 0 = active (2020-02-01), 1 = churned (other dates)
data_contract['EndDate'] = data_contract['EndDate'].apply(lambda x: 0 if x == pd.Timestamp('2020-02-01') else 1)

    #!! data_contract columns renamed
data_contract.rename(columns={'EndDate': 'churned', 'Type': 'contract_type', 'PaperlessBilling': 'paperless_billing',
                              'PaymentMethod': 'payment_method', 'MonthlyCharges': 'monthly_charges'}, inplace=True)

In [3]:
    #!! data_internet columns renamed
data_internet.rename(columns={'InternetService': 'internet_service', 'OnlineSecurity': 'online_security',
                              'OnlineBackup': 'online_backup', 'DeviceProtection': 'device_protection',
                              'TechSupport': 'tech_support', 'StreamingTV': 'streaming_tv',
                              'StreamingMovies': 'streaming_movies'}, inplace=True)

In [4]:
    #!! dropping gender column (no relationship with customer loyalty)
data_personal = data_personal.drop('gender', axis=1)

    #!! data_personal columns renamed
data_personal.rename(columns={'SeniorCitizen': 'senior_citizen', 'Partner': 'partner',
                              'Dependents': 'dependents'}, inplace=True)

In [5]:
    #!! data_phone columns renamed
data_phone.rename(columns={'MultipleLines': 'multiple_lines'}, inplace=True)

In [6]:
#data_contract.info()
#data_internet.info()
#data_personal.info()
#data_phone.info()

display(data_contract.head())
display(data_internet.head())
display(data_personal.head())
display(data_phone.head())

    #!! checking unique values
#for col in data_contract.select_dtypes(include=['object', 'int64']).columns: display(data_contract[col].value_counts())

    #!! checking unique values
#for col in data_internet.select_dtypes(include=['object']).columns: display(data_internet[col].value_counts())

    #!! checking unique values
#for col in data_personal.select_dtypes(include=['object', 'int64']).columns: display(data_personal[col].value_counts())

    #!! checking unique values
#for col in data_phone.select_dtypes(include=['object']).columns: display(data_phone[col].value_counts())

,customerID,churned,contract_type,paperless_billing,payment_method,monthly_charges,tenure_month
0,7590-VHVEG,0,Month-to-month,Yes,Electronic check,29.85,1
1,5575-GNVDE,0,One year,No,Mailed check,56.95,34
2,3668-QPYBK,1,Month-to-month,Yes,Mailed check,53.85,2
3,7795-CFOCW,0,One year,No,Bank transfer (automatic),42.30,45
4,9237-HQITU,1,Month-to-month,Yes,Electronic check,70.70,2


,customerID,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies
0,7590-VHVEG,DSL,No,Yes,No,No,No,No
1,5575-GNVDE,DSL,Yes,No,Yes,No,No,No
2,3668-QPYBK,DSL,Yes,Yes,No,No,No,No
3,7795-CFOCW,DSL,Yes,No,Yes,Yes,No,No
4,9237-HQITU,Fiber optic,No,No,No,No,No,No


,customerID,senior_citizen,partner,dependents
0,7590-VHVEG,0,Yes,No
1,5575-GNVDE,0,No,No
2,3668-QPYBK,0,No,No
3,7795-CFOCW,0,No,No
4,9237-HQITU,0,No,No


,customerID,multiple_lines
0,5575-GNVDE,No
1,3668-QPYBK,No
2,9237-HQITU,No
3,9305-CDSKC,Yes
4,1452-KIOVK,Yes


In [7]:
    #!! merging all dataframes on customerID
data = data_contract.merge(
    data_internet, on='customerID', how='left').merge(
        data_personal, on='customerID', how='left').merge(
            data_phone, on='customerID', how='left')

#data.info()
#display(data.head())
#display(data['multiple_lines'].value_counts())

    #!! filling 'No Service' for NaN values (customers without internet or phone service)
data.fillna('No Service', inplace=True)

    #!! creating binary columns for internet and phone service (0 = No Service, 1 = has service)
data['has_internet'] = data['internet_service'].apply(lambda x: 0 if x == 'No Service' else 1)
data['has_phone'] = data['multiple_lines'].apply(lambda x: 0 if x == 'No Service' else 1)

#data.info()
display(data.head())
#display(data['multiple_lines'].value_counts())

,customerID,churned,contract_type,paperless_billing,payment_method,monthly_charges,tenure_month,internet_service,online_security,online_backup,device_protection,tech_support,streaming_tv,streaming_movies,senior_citizen,partner,dependents,multiple_lines,has_internet,has_phone
0,7590-VHVEG,0,Month-to-month,Yes,Electronic check,29.85,1,DSL,No,Yes,No,No,No,No,0,Yes,No,No Service,1,0
1,5575-GNVDE,0,One year,No,Mailed check,56.95,34,DSL,Yes,No,Yes,No,No,No,0,No,No,No,1,1
2,3668-QPYBK,1,Month-to-month,Yes,Mailed check,53.85,2,DSL,Yes,Yes,No,No,No,No,0,No,No,No,1,1
3,7795-CFOCW,0,One year,No,Bank transfer (automatic),42.30,45,DSL,Yes,No,Yes,Yes,No,No,0,No,No,No Service,1,0
4,9237-HQITU,1,Month-to-month,Yes,Electronic check,70.70,2,Fiber optic,No,No,No,No,No,No,0,No,No,No,1,1


**data cleaning finished, onto data split and preprocessing. (OHE, StandardScaler)**